# LLM-Powered Structured Insight & RAG Retrieval Pipeline — Demo Notebook

Run each cell **one by one, in order**. This notebook demonstrates:
1. Structured extraction with Pydantic validation (including a deliberate validation failure test)
2. The full RAG pipeline — chunking, embedding, vector storage, retrieval, and grounded generation

**Prerequisites:**
- `pip install -r requirements.txt` already run
- `GEMINI_API_KEY` environment variable set (or set it in Cell 1 below)
- Run from the project root so relative paths (`data/`, `src/`, `results/`) resolve correctly


## Step 0 — Setup: API Key & Path

In [ ]:
import os
import sys

# If GEMINI_API_KEY is not already set in your environment, uncomment and set it here:
# os.environ["GEMINI_API_KEY"] = "your-free-gemini-api-key"

assert os.environ.get("GEMINI_API_KEY"), "GEMINI_API_KEY not set. Export it in your shell or set it above."

# Make sure src/ is importable from the notebook
sys.path.append(os.path.abspath("src"))
print("Environment ready.")


## Step 1 — Generate the Sample Support Tickets Dataset

In [ ]:
!python scripts/create_sample_data.py


## Step 2 — Structured Extraction: Validation Failure Test

Runs the deliberate malformed fixture (`urgency: "SUPER_URGENT"`) through `ExtractedSupportTicket` to confirm Pydantic's enum validation rejects invalid values.

In [ ]:
from schemas import ExtractedSupportTicket
from pydantic import ValidationError

malformed_fixture = {
    "ticket_id": "TCK-INVALID-01",
    "category": "Billing",
    "urgency": "SUPER_URGENT",  # Invalid Enum Value
    "sentiment": "Negative",
    "one_line_summary": "User wants a refund."
}

try:
    ExtractedSupportTicket(**malformed_fixture)
    print("Test Failed: Malformed fixture passed unexpectedly.")
except ValidationError as e:
    print("Successfully Caught Validation Failure!")
    print(f"Validation Error Details:\n{e}")


## Step 3 — Run Full Extraction Pipeline on Sample Tickets

Calls Gemini with structured output enforcement for each ticket in `data/support_tickets.json`, validates every response against the Pydantic schema, and saves results to `results/extracted_tickets.json`.

In [ ]:
!python src/extraction_pipeline.py


In [ ]:
import json

with open("results/extracted_tickets.json") as f:
    extracted = json.load(f)

print(f"Validated {len(extracted)} tickets.\n")
for t in extracted[:3]:
    print(t)


## Step 4 — Run the RAG Pipeline

Chunks `data/knowledge_base.txt`, embeds the chunks with `all-MiniLM-L6-v2`, stores them in an in-memory ChromaDB collection, retrieves the most relevant chunks per query, and generates a grounded answer using Gemini.

In [ ]:
!python src/rag_pipeline.py


## Step 5 — (Optional) Interactive Query

Run a custom query against the knowledge base directly in the notebook, without re-running the full CLI script.

In [ ]:
from rag_pipeline import chunk_document, generate_embeddings, populate_vector_store, retrieve_relevant_chunks, generate_grounded_answer

with open("data/knowledge_base.txt", "r", encoding="utf-8") as f:
    corpus_text = f.read()

chunks = chunk_document(corpus_text, chunk_size=300, overlap=50)
embedder, embeddings = generate_embeddings(chunks)
collection = populate_vector_store(chunks, embeddings)

my_query = "What is the SSO outage resolution process for Enterprise customers?"
retrieved = retrieve_relevant_chunks(my_query, embedder, collection, k=2)
answer = generate_grounded_answer(my_query, retrieved)

print("Query:", my_query)
print("\nRetrieved Chunks:")
for c in retrieved:
    print("-", c[:200], "...\n")
print("Generated Answer:\n", answer)


## Summary

- Section 2 confirmed schema validation correctly rejects malformed enum values.
- Section 3 produced validated, structured extractions for all 15 sample tickets.
- Section 4 ran the 5 benchmark queries against the internal knowledge base with grounded, source-backed answers (see `demo_transcript.md` for the full recorded transcript).
- Section 5 demonstrated ad-hoc querying beyond the fixed test set.
